In [ ]:
import sys
import os
from langchain.chat_models import init_chat_model

from dotenv import load_dotenv

load_dotenv(override=True)

In [ ]:
LLM_MODEL = os.getenv("LLM_MODEL")
LLM_BASE_URL=os.getenv("LLM_BASE_URL")
LLM_API_KEY=os.getenv("LLM_API_KEY")
LLM_TEMPERATURE=os.getenv("LLM_TEMPERATURE")

# vLLM 모델 인스턴스 생성
llm = init_chat_model(
    "openai:" + LLM_MODEL,
    temperature=LLM_TEMPERATURE,
    base_url=LLM_BASE_URL,
    api_key=LLM_API_KEY
)

In [3]:
import os
from pathlib import Path
from typing import List

def table_to_markdown(table: List[List]) -> str:
    """
    표 데이터를 마크다운 테이블 형식으로 변환하는 헬퍼 함수
    
    Args:
        table: 2차원 리스트 형태의 표 데이터
    
    Returns:
        마크다운 테이블 문자열
    """
    if not table or len(table) == 0:
        return ""
    
    # 빈 셀을 빈 문자열로 변환
    def clean_cell(cell):
        if cell is None:
            return ""
        return str(cell).strip()
    
    # 표 데이터 정리
    cleaned_table = [[clean_cell(cell) for cell in row] for row in table]
    
    # 최대 컬럼 수 확인
    max_cols = max(len(row) for row in cleaned_table) if cleaned_table else 0
    
    # 모든 행을 동일한 컬럼 수로 맞춤
    normalized_table = []
    for row in cleaned_table:
        normalized_row = row + [""] * (max_cols - len(row))
        normalized_table.append(normalized_row)
    
    if not normalized_table:
        return ""
    
    markdown_lines = []
    
    # 헤더 행 (첫 번째 행)
    header = normalized_table[0]
    markdown_lines.append("| " + " | ".join(header) + " |")
    
    # 구분선
    markdown_lines.append("| " + " | ".join(["---"] * len(header)) + " |")
    
    # 데이터 행들
    for row in normalized_table[1:]:
        markdown_lines.append("| " + " | ".join(row) + " |")
    
    return "\n".join(markdown_lines)


def extract_text_from_pdf(pdf_path: str, password: str = None) -> str:
    """
    PDF 파일을 마크다운 형식으로 변환하여 반환하는 함수
    
    표는 마크다운 테이블 형식으로 변환되고, 텍스트는 그대로 유지됩니다.
    
    Args:
        pdf_path: PDF 파일 경로 (상대 경로 또는 절대 경로)
        password: 암호화된 PDF의 비밀번호 (선택사항)
    
    Returns:
        마크다운 형식으로 변환된 문자열 (텍스트 + 표)
    
    Raises:
        FileNotFoundError: PDF 파일을 찾을 수 없을 때
        ImportError: 필요한 PDF 라이브러리가 설치되지 않았을 때
        ValueError: PDF 암호가 틀렸을 때
    """
    # 파일 경로 확인 및 절대 경로로 변환
    pdf_path = Path(pdf_path)
    if not pdf_path.is_absolute():
        # 노트북 위치 기준 상대 경로 처리
        # 노트북은 asset_ai_portal/tests 폴더에 있고, documents는 20_code_test 루트에 있음
        current_dir = Path.cwd()
        
        # asset_ai_portal/tests에서 실행 중이면 상위로 두 번 이동 (20_code_test 루트)
        if current_dir.name == 'tests' and current_dir.parent.name == 'asset_ai_portal':
            project_root = current_dir.parent.parent  # tests -> asset_ai_portal -> 20_code_test
        elif current_dir.name == 'asset_ai_portal':
            project_root = current_dir.parent  # asset_ai_portal -> 20_code_test
        else:
            # 20_code_test에서 실행 중이면 그대로 사용
            project_root = current_dir
        
        pdf_path = project_root / pdf_path
    
    if not pdf_path.exists():
        raise FileNotFoundError(f"PDF 파일을 찾을 수 없습니다: {pdf_path}")
    
    # 여러 PDF 라이브러리 시도 (우선순위 순)
    # 1. pdfplumber (표 추출에 유리, 암호화된 PDF 지원, 마크다운 변환에 최적)
    try:
        import pdfplumber
        
        markdown_parts = []
        with pdfplumber.open(str(pdf_path), password=password) as pdf:
            for page_num, page in enumerate(pdf.pages, 1):
                page_content = []
                
                # 표 추출 (표가 있으면 먼저 표를 추출)
                tables = page.extract_tables()
                if tables:
                    for table_idx, table in enumerate(tables):
                        if table:
                            markdown_table = table_to_markdown(table)
                            if markdown_table:
                                page_content.append(markdown_table)
                                page_content.append("")  # 표 다음에 빈 줄 추가
                
                # 텍스트 추출
                text = page.extract_text()
                if text:
                    # 표와 겹치는 텍스트를 제거하기 위해 간단한 필터링
                    # (실제로는 더 정교한 로직이 필요할 수 있음)
                    page_content.append(text)
                
                if page_content:
                    markdown_parts.append("\n".join(page_content))
        
        return "\n\n".join(markdown_parts) if markdown_parts else ""
    except ImportError:
        pass
    except Exception as e:
        # 암호화 관련 오류인지 확인
        error_msg = str(e).lower()
        if password and ('password' in error_msg or 'encrypted' in error_msg or 'incorrect password' in error_msg):
            raise ValueError(f"PDF 암호가 올바르지 않거나 암호화된 PDF를 읽을 수 없습니다: {e}")
        # 암호화되지 않은 PDF인데 비밀번호가 필요한 경우는 다음 라이브러리로 시도
        if not password and ('encrypted' in error_msg or 'password' in error_msg):
            raise ValueError("PDF가 암호화되어 있습니다. 비밀번호를 제공해주세요.")
        # 다른 오류는 다음 라이브러리로 시도
        pass
    
    # 2. pypdf (가장 가벼움, 암호화된 PDF 지원, 표 추출 불가)
    try:
        from pypdf import PdfReader
        reader = PdfReader(str(pdf_path), password=password)
        # pypdf는 password를 전달하면 자동으로 암호 해제를 시도합니다
        # 암호가 틀렸거나 암호화된 PDF인데 비밀번호가 없으면 예외가 발생합니다
        text_parts = []
        for page in reader.pages:
            text = page.extract_text()
            if text:
                text_parts.append(text)
        return "\n".join(text_parts)
    except ImportError:
        pass
    except Exception as e:
        # 암호화 관련 오류인지 확인
        error_msg = str(e).lower()
        if password and ('password' in error_msg or 'encrypted' in error_msg or 'incorrect password' in error_msg):
            raise ValueError(f"PDF 암호가 올바르지 않거나 암호화된 PDF를 읽을 수 없습니다: {e}")
        # 암호화되지 않은 PDF인데 비밀번호가 필요한 경우는 다음 라이브러리로 시도
        if not password and ('encrypted' in error_msg or 'password' in error_msg):
            raise ValueError("PDF가 암호화되어 있습니다. 비밀번호를 제공해주세요.")
        # 다른 오류는 다음 라이브러리로 시도
        pass
    
    # 3. PyPDF2 (구버전 호환, 암호화된 PDF 지원)
    try:
        import PyPDF2
        with open(pdf_path, 'rb') as file:
            reader = PyPDF2.PdfReader(file)
            if reader.is_encrypted:
                if not password:
                    raise ValueError("PDF가 암호화되어 있습니다. 비밀번호를 제공해주세요.")
                if not reader.decrypt(password):
                    raise ValueError("PDF 암호가 올바르지 않습니다.")
            text_parts = []
            for page in reader.pages:
                text_parts.append(page.extract_text())
            return "\n".join(text_parts)
    except ImportError:
        pass
    except ValueError as e:
        # 암호 오류는 그대로 전파
        raise
    except Exception as e:
        # 암호화 관련 오류인지 확인
        if password and ('password' in str(e).lower() or 'encrypted' in str(e).lower()):
            raise ValueError(f"PDF 암호가 올바르지 않거나 암호화된 PDF를 읽을 수 없습니다: {e}")
        # 다른 오류는 무시하고 다음 단계로
        pass
    
    # 모든 라이브러리가 없으면 에러
    raise ImportError(
        "PDF 텍스트 추출을 위한 라이브러리가 설치되지 않았습니다. "
        "다음 중 하나를 설치해주세요: pdfplumber, pypdf, PyPDF2\n"
        "표 추출 기능을 사용하려면 pdfplumber를 설치하는 것을 권장합니다.\n"
        "설치 명령: pip install pdfplumber 또는 pip install pypdf"
    )


# 사용 예시: 대신.pdf 파일 텍스트 추출
# pdf_file_path = "documents/sample_overseas_settlement/SB(Bernstein).pdf"
pdf_file_path = "documents/sample_variable_annuity/신한라이프(퇴직)_251127.pdf"
try:
    pdf_text = extract_text_from_pdf(pdf_file_path, "345678")
    print(f"✅ PDF 텍스트 추출 완료 ({len(pdf_text)} 문자)")
    print("\n" + "="*80)
    print("추출된 텍스트:")
    print("="*80)
    print(pdf_text)
    
except Exception as e:
    print(f"❌ 오류 발생: {e}")


✅ PDF 텍스트 추출 완료 (2036 문자)

추출된 텍스트:
| 기준일자 : 2025-11-27 |  |  |  | 수신처 : 삼성액티브자산운용 |  |  |  |  |  |  | CUTOFF대상여부 : N |  |  |  |  |  |  |  |  |  |  |  |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 통합
펀드코드 | 서브
펀드코드 |  | 펀드명 |  |  | 운용사 | 입금액 |  | 출금액 |  |  | 당일이체좌수 |  | 당일이체금액 |  | 이체 예정금액 |  |  |  |  |  |  |
|  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  | 2025-11-28 |  | 2025-12-01 |  | 2025-12-02 |  | 2025-12-03 |
| DV004 | M0402 |  | 퇴직혼합형-혼합형 |  |  | 삼성액티브자산운용 | 0 |  | 0 |  |  | 0 |  | -19,267,686 |  | 846,032 |  | -121,762,204 |  | 0 |  | 0 |
| DV005 | M0501 |  | 퇴직 주식형-주식형1 |  |  | 삼성액티브자산운용 | 0 |  | 0 |  |  | 0 |  | 0 |  | 1,157,261 |  | 0 |  | 0 |  | 0 |
| 설정 합계 |  |  |  |  |  |  |  |  |  |  |  | 0 |  | 0 |  | 2,003,293 |  | 0 |  | 0 |  | 0 |
| 해지 합계 |  |  |  |  |  |  |  |  |  |  |  | 0 |  | -19,267,686 |  | 0 |  | -121,762,204 |  | 0 |  | 0 |
| 총 계 |  |  |  |  |  |  |  |  |  |  

In [ ]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage

# 메시지 객체 생성
system_msg = SystemMessage("당신은 자산운용사에서 해외거래체결 확인을 담당하는 오퍼레이터 입니다.")
human_msg = HumanMessage(f"""
아래는 브로커가 보내온 해외거래체결내역 메일입니다.
메일 내용을 분석하여 해외거래체결내역 대사 업무를 위한 거래 체결 정보를 수집하세요.

** 반드시 지켜야 할 중요한 사항 **
1. 모든 종목을 전부 수집하세요.(주요 종목만 수집하면 안됩니다.)

### 해외거래체결내역 메일 내용 ###
{pdf_text}
""")

# 채팅 모델과 함께 사용
messages = [system_msg, human_msg]
response = llm.invoke(messages)  # AIMessage 반환

In [ ]:
from IPython.display import Markdown, display

# LLM 응답을 마크다운 형식으로 보기 좋게 표시
if 'response' in locals():
    display(Markdown(response.content))
    
    # 추가 정보 (토큰 사용량 등)를 표시
    if hasattr(response, 'response_metadata') and response.response_metadata:
        metadata = response.response_metadata
        if 'token_usage' in metadata:
            print("\n---")
            print("**토큰 사용량:**")
            print(f"- 입력 토큰: {metadata['token_usage'].get('prompt_tokens', 'N/A')}")
            print(f"- 출력 토큰: {metadata['token_usage'].get('completion_tokens', 'N/A')}")
            print(f"- 총 토큰: {metadata['token_usage'].get('total_tokens', 'N/A')}")
else:
    print("⚠️ 'response' 변수를 찾을 수 없습니다. 먼저 LLM을 호출해주세요.")
